# Knowledge Agent — Agentic Knowledge & Retrieval
### Business Demo

---

## What is the Knowledge Agent?

The **Knowledge Agent** is the agentic system's information retrieval layer. It answers the same questions as the nonagent Level 1 system — but with key architectural differences:

- **Orchestrator Agent** classifies intent and routes to the Knowledge Agent dynamically
- **MCP Protocol** — tools are invoked via `CustomerDataServer` and `ProductCatalogServer` (not direct function calls)
- **Memory Layer** — every interaction is stored in conversation + interaction memory for context
- **Agent History** — full execution path is recorded (`orchestrator → knowledge`)

### How it works (agentic flow)

```
Your question
     │
     ▼
Orchestrator Agent: Classify intent → "knowledge"
     │
     ▼
Knowledge Agent: Select MCP server based on query type
     │
     ├── customer_data_server.search_customer_profile
     ├── customer_data_server.get_identity_status
     └── knowledge_base.multi_query_search (emails, notes, policies)
     │
     ▼
Memory Layer: Record to conversation + interaction memory
     │
     ▼
Answer returned with MCP call trace + agent history
```

### Agentic vs Nonagent — Key Differences

| Feature | Nonagent Level 1 | Knowledge Agent |
|---------|-----------------|----------------|
| **Routing** | Fixed intent → level mapping | Orchestrator classifies dynamically |
| **Tool Access** | Direct function calls | MCP protocol (server abstraction) |
| **Memory** | Stateless | Conversation + interaction memory |
| **Traceability** | `tool_calls` list | `mcp_calls` + `agent_history` |
| **Autonomy** | Low | High (orchestrator decides) |

---

## Setup

Run this cell once to initialise the agentic system.

In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

_root = Path(os.getcwd())
while not (_root / "nonagentic" / "__init__.py").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
load_dotenv()
os.environ["GUARDRAIL_ENABLED"] = "false"

from notebooks.agentic.utils.helpers import (
    agentic_ask,
    show_agentic_result,
)
from notebooks.agentic.utils.helpers import (
    display_agentic_customer_profile,
    display_agentic_identity_analysis,
    display_agentic_search_results,
    display_outcome_analysis,
    display_agentic_policy_qa,
    display_agentic_audit_trail,
    display_agentic_source_breakdown,
    run_agentic_guardrail_demo,
)


from notebooks.nonagentic.utils.demo_data import get_demo_customers, get_test_queries
from notebooks.nonagentic.utils.display_helpers import display_metrics

print("Agentic System Ready | Project root:", _root)

Agentic System Ready | Project root: /home/kahloka/projects/agenticAI


---
## Use Case 1 — Customer Profile Lookup

**Business scenario**: An account manager needs a quick overview of a customer before a call.

**Agentic difference**: The Orchestrator classifies this as `knowledge` intent and routes to the Knowledge Agent, which invokes `customer_data_server.search_customer_profile` via MCP. The interaction is stored in memory.

In [2]:
customer_id = "CUST-005"
result = agentic_ask(f"What is the profile for customer {customer_id}?", customer_id=customer_id)
show_agentic_result(result, title="Customer Profile Query")
display_agentic_customer_profile(result)

,Category
0,automotive
1,clothing
2,toys


### Agentic Extras — Agent Path & MCP Calls

Unlike the nonagent system, we can inspect exactly which agents ran and which MCP tools were invoked.

In [3]:

display_metrics({
    "Agent Path": " → ".join(result.get("agent_history", [])),
    "MCP Server": result["mcp_calls"][0]["server"] if result.get("mcp_calls") else "none",
    "MCP Tool": result["mcp_calls"][0]["tool"] if result.get("mcp_calls") else "none",
})

# Show memory was recorded
from memory.memory_manager import memory_manager
interactions = memory_manager.interaction.get_interactions(result.get("user_id", "manager@shop.com"))
print(f"Memory: {len(interactions)} interaction(s) recorded for this session")

Memory: 1 interaction(s) recorded for this session


---
## Use Case 2 — Identity Verification Status

**Business scenario**: Trust & Safety needs to verify identity status before approving high-value purchases.

We check **15 customers** with a realistic distribution: verified (60%), unverified (27%), pending (13%).

**Agentic note**: This reads directly from the database for batch display — same data source as the nonagent system.

In [4]:
demo_customers = get_demo_customers()
customers_to_check = demo_customers["identity_verification_mix"]
identity_results = display_agentic_identity_analysis(customers_to_check)

Customer,Status,Fraud Score
CUST-001,pending,0.126
CUST-004,verified,0.366
CUST-006,verified,0.260
CUST-007,pending,0.171
CUST-008,pending,0.179
CUST-009,unverified,0.126
CUST-011,pending,0.188
CUST-012,unverified,0.353
CUST-015,verified,0.379
CUST-005,verified,0.183


### Agentic: Single Customer Identity via Knowledge Agent

For individual lookups, the Knowledge Agent uses `customer_data_server.get_identity_status` via MCP.

In [5]:
result = agentic_ask("What is the identity verification status for CUST-001?", customer_id="CUST-001")
show_agentic_result(result, title="Identity Status — CUST-001")

display_metrics({
    "Agent": result.get("active_agent", "unknown"),
    "Intent": result.get("intent", "unknown"),
    "MCP Tool Used": result["mcp_calls"][0]["tool"] if result.get("mcp_calls") else "knowledge_base",
})

---
## Use Case 3 — Email Correspondence Search

**Business scenario**: A customer claims they never received a renewal reminder. The system searches the email archive semantically.

**Agentic difference**: The Knowledge Agent routes to `knowledge_base.multi_query_search` (no customer_id in query), which searches the ChromaDB email collection. The result is stored in interaction memory.

In [6]:
# 'What ... exist' phrasing routes reliably to knowledge_agent

result = agentic_ask("What emails exist about identity verification renewal reminders?")
display_agentic_search_results(result, "Email Search Results", emoji="📧", color="#8b5cf6")

Source,Type,Relevance,Preview
identity-verification-policy.md,,0.699,...
CUST-058-identity_verification_reminder.eml,,0.707,...
CUST-060-identity_verification_reminder.eml,,0.743,...
CUST-029-identity_verification_reminder.eml,,0.804,...
CUST-152-identity_verification_reminder.eml,,0.812,...


---
## Use Case 4 — CRM Notes Search

**Business scenario**: A senior manager wants to understand escalation history before a customer review meeting.

The Knowledge Agent searches the notes ChromaDB collection using semantic similarity — same underlying tool as the nonagent system, but invoked via MCP protocol.

In [7]:
# 'Search for' phrasing routes correctly to knowledge_agent
result1 = agentic_ask("Search for agent notes about customer escalations and complaints")
display_agentic_search_results(result1, "Escalations & Complaints", emoji="⚠️", color="#ef4444")

result2 = agentic_ask("Search for agent notes about customer retention and loyalty bonuses")
display_agentic_search_results(result2, "Retention & Loyalty", emoji="🎁", color="#10b981")

# Outcome analysis on retention chunks
chunks2 = (result2.get("final_result") or {}).get("chunks", [])
if chunks2:
    display_outcome_analysis(chunks2, "📊 Loyalty Bonus Outcomes")

Source,Type,Relevance,Preview
CUST-087-note.txt,,0.965,...
CUST-206-note.txt,,0.979,...
CUST-042-note.txt,,0.980,...
CUST-037-note.txt,,0.982,...
CUST-030-note.txt,,0.984,...


Source,Type,Relevance,Preview
CUST-033-note.txt,,0.901,...
CUST-173-note.txt,,0.912,...
CUST-043-note.txt,,0.920,...
CUST-047-note.txt,,0.923,...
CUST-164-note.txt,,0.929,...


---
## Use Case 5 — Policy Document Search (RAG)

**Business scenario**: A product manager needs to check eligibility rules or a trust & safety officer needs to verify the identity verification policy.

The Knowledge Agent uses **Retrieval-Augmented Generation (RAG)**:
1. Routes to `knowledge_base.multi_query_search` on the policy collection
2. LLM synthesises a cited answer from retrieved policy sections
3. Interaction is recorded to memory for follow-up context

**Agentic advantage**: Memory means follow-up questions can reference prior policy answers.

In [8]:

queries = get_test_queries()
display_agentic_policy_qa(queries["policy_questions"], "Policy Document Search (RAG)")

---
## Use Case 6 — Cross-Source Search

**Business scenario**: A manager asks a broad question spanning policies, emails, and notes. The system searches all collections simultaneously.

**Agentic difference**: The Knowledge Agent's `multi_query_search` searches all ChromaDB collections in one call. The source breakdown shows which collections contributed — and the MCP call trace shows exactly what was invoked.

In [9]:
# 'Search policies and notes' phrasing routes reliably to knowledge_agent
result = agentic_ask("Search policies and notes about return risk and churn prevention")

display_agentic_search_results(result, "Cross-Source Answer", emoji="🔍", color="#8b5cf6")
display_agentic_source_breakdown(result, "Sources by Collection")

Source,Type,Relevance,Preview
CUST-164-note.txt,,1.063,...
CUST-115-note.txt,,1.068,...
fraud-policy.md,,1.081,...
CUST-202-note.txt,,1.181,...
CUST-207-note.txt,,1.188,...


---
## Use Case 7 — Audit Trail

**Business scenario**: Trust & Safety requires a full record of every data access.

**Agentic advantage over nonagent**: The agentic system records three layers of traceability:
1. `audit_trail` — agent-level actions (same as nonagent)
2. `mcp_calls` — every MCP tool invocation with server, tool name, and params
3. `memory` — interaction history stored in the memory layer across sessions

In [10]:
result = agentic_ask(
    "What is the profile for CUST-005?",
    customer_id="CUST-005",
    user_id="trustsafety@shop.com"
)
display_agentic_audit_trail(result)

agent,action,intent,confidence,next_agent,mcp_calls
orchestrator,classify_and_plan,knowledge,0.800000,knowledge,nan
knowledge,retrieve_information,nan,nan,nan,1.000000


Server,Tool,Params
customer_data,search_customer_profile,{'customer_id': 'CUST-005'}


### Memory Layer — Interaction History

Unlike the nonagent system, the agentic system persists interactions in memory across calls within a session.

In [12]:
from memory.memory_manager import memory_manager
import pandas as pd
from IPython.display import display, HTML

interactions = memory_manager.interaction.get_interactions("trustsafety@shop.com")
observations = memory_manager.agent_observation.get_observations("orchestrator")

display_metrics({
    "Interactions Recorded": len(interactions),
    "Orchestrator Observations": len(observations),
})

if interactions:
    display(HTML('<div style="font-size:16px;font-weight:600;margin:16px 0 8px 0">📝 Recent Interactions in Memory</div>'))
    rows = [{"Type": i.get("type", ""), "Timestamp": i.get("timestamp", "")[:19]} for i in interactions[-5:]]
    df = pd.DataFrame(rows)
    display(df.style.set_table_styles([
        {'selector': 'th', 'props': [('background', '#667eea'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #e2e8f0')]},
    ]).hide(axis='index'))

Type,Timestamp
knowledge_retrieval,2026-03-13T01:17:45


---
## Use Case 8 — Output Guardrails

**Business scenario**: Before any answer reaches a user, it passes through an automatic guardrail check.

- **PII leakage** — email addresses and phone numbers are automatically redacted
- **Forbidden content** — phrases like *"financial advice"* or *"buy shares"* are blocked

The guardrail is identical to the nonagent system — it runs on the output regardless of which agent produced it.

In [13]:
from IPython.display import display, HTML

display(HTML('<div style="font-size:18px;font-weight:600;margin:16px 0">🛡️ Guardrail Check</div>'))
run_agentic_guardrail_demo()

---
## Summary

| Use Case | What the system does | Agentic layer |
|---|---|---|
| **1. Customer Profile** | Returns full profile via MCP `search_customer_profile` | `customer_data_server` |
| **2. Identity Verification** | Status lookup via MCP `get_identity_status` | `customer_data_server` |
| **3. Email Search** | Semantic search via `multi_query_search` | `knowledge_base` (ChromaDB) |
| **4. CRM Notes** | Escalation/retention notes via semantic search | `knowledge_base` (ChromaDB) |
| **5. Policy Search** | RAG over policy library with cited answers | `knowledge_base` (ChromaDB) |
| **6. Cross-Source** | All collections searched simultaneously | `knowledge_base` (all) |
| **7. Audit Trail** | `audit_trail` + `mcp_calls` + memory layer | Memory Manager |
| **8. Guardrails** | PII redacted, forbidden content blocked | Inline guardrail check |

---

### Agentic advantages over nonagent Level 1

- **MCP abstraction** — tools are invoked via server protocol, not direct function calls
- **Memory** — every interaction stored; follow-up questions have full context
- **Agent history** — full execution path visible (`orchestrator → knowledge`)
- **Dynamic routing** — Orchestrator classifies intent with LLM, not fixed rules
- **Observations** — Orchestrator records learnings to `agent_observation` memory

---

### What comes next

- **Analytics Agent** — SQL queries, segmentation, insights (agentic counterpart of Level 2)
- **Recommendation Agent** — personalized offers with memory context
- **Workflow Agent** — multi-step goals with evaluation and replanning